# Task 2, 3 & 4: Idiom-Aware Identification

This notebook refines the identification process by applying specific rules for each visualization type (A, B, C, D) and captures the hierarchical parent-child relationships.

In [3]:
import xml.etree.ElementTree as ET
import os
import re

class AOIPipeline:
    def __init__(self):
        # STEP 1: SPECIFIC RULES PER TYPE / IDIOM
        # Based on manual audit of final stimuli SVGs
        self.idiom_rules = {
            'Aa': [
                'Sub-regions', 'data-area-container', 'area-', 'region-',
                'area-labels', 'label-container', 'label-text',
                'legend-', 'color-'
            ],
            'Ab': [
                'treemap_root', 'leaf-', 'label-container', 'label-text'
            ],

            'Ba': [
                'reference-axis', 'axisLine', 'axisTicks',
                'axis-label-container', 'axis-title', 'reference-grid',
                'data-groups',
                'legend-container', 'color-legend-container', 'legend-color-item',
                'shape-legend-container', 'legend-shape-item',
            ],

            'Bb': [
                'reference-axis', 'axisLine', 'axisTicks',
                'axis-title', 'reference-grid', 'main-grid',
                'data-points',
                'legend-container', 'color-legend-container', 'legend-color-item',
            ],
            'Ca': [
                'legend-container', 'colorbins-', 'colorbin-', 'labels-container',
                'data-container', 'xAxis', 'yAxis', 'cell-', 'label-'
            ],
            'Cb': [
                'legend-container', 'colorbins-', 'colorbin-', 'labels-container',
                'data-container', 'xAxis', 'yAxis', 'cell-', 'label-'
            ],
            'Da': [
                'legend', 'legend-item', 'lines-paths', 'line-',
                'all-stars', 'circle-', 'label-'
            ],
            'Db': [
                'legend-container', 'legend-item', 'data-container',
                'links', 'link_', 'nodes', 'node-', 'node-labels', 'label-'
            ]
        }

    def _get_type_info(self, file_path):
        """
        Detects the stimulus_id, main type, and subtype from the filename.
        Example: A-a-1.svg -> {stim_id: 'A-a-1', viz_type: 'Aa'}
        """
        filename = os.path.basename(file_path)
        stim_id = filename.replace(".svg", "")
        parts = stim_id.split("-")

        if len(parts) >= 2:
            main_type = parts[0].upper()   # A, B, C, D
            subtype = parts[1].lower()     # a, b
            return stim_id, main_type + subtype

        return stim_id, filename[0].upper()

    def _is_meaningful(self, elem_id, elem_class, parent_id, prefixes):
        """
        Checks whether an SVG element is meaningful based on:
        - id
        - class
        - parent id
        """
        combined = f"{elem_id} {elem_class} {parent_id}".lower()

        for p in prefixes:
            p_lower = p.lower()

            if elem_id == p:
                return True

            if elem_id.startswith(p):
                return True

            if p_lower in combined:
                return True

        return False

    def _classify_semantic_type(self, elem_id, elem_class, parent_id, tag):
        """
        Categorizes elements into data_mark, legend, axis, or label.
        """
        combined = f"{elem_id} {elem_class} {parent_id}".lower()

        if "legend" in combined or "color-" in combined or "colorbin" in combined:
            return "legend"
        if "label" in combined or tag == "text":
            return "label"
        if "axis" in combined or "xaxis" in combined or "yaxis" in combined:
            return "axis"

        # Default data marks
        data_mark_triggers = ["area-", "region-", "leaf-", "cell-", "node-", "link_", "circle-", "line-"]
        if any(t in combined for t in data_mark_triggers) or "data-points" in combined:
            return "data_mark"

        if tag == "g":
            return "group"

        return "unknown"

    def _get_family(self, semantic_type):
        """
        Higher-level grouping as requested by the Goldmine file.
        """
        mapping = {
            "legend": "legend",
            "label": "annotations",
            "axis": "chart_frame",
            "data_mark": "data_marks",
            "group": "structural"
        }
        return mapping.get(semantic_type, "other")

    def _get_clean_label(self, elem_id, text_content):
        """
        Attempts to create a human-readable label.
        """
        if text_content:
            return text_content

        # Clean ID (e.g., area-RWA_1_1_Redwood_Hills -> Redwood Hills)
        clean = elem_id.split("-")[-1]
        clean = clean.replace("_", " ")
        return clean

    def _get_text_content(self, elem):
        """
        Extracts text content from SVG text elements.
        """
        text_parts = []
        for child in elem.iter():
            if child.text:
                text_parts.append(child.text.strip())
        return " ".join([t for t in text_parts if t])

    def _get_element_bounds(self, elem):
        coords = []
        tag = elem.tag.split('}')[-1]

        if tag == 'rect':
            x, y = float(elem.get('x', 0)), float(elem.get('y', 0))
            w, h = float(elem.get('width', 0)), float(elem.get('height', 0))
            coords.extend([(x, y), (x + w, y + h)])
            
        elif tag == 'circle':
            cx, cy = float(elem.get('cx', 0)), float(elem.get('cy', 0))
            r = float(elem.get('r', 0))
            coords.extend([(cx - r, cy - r), (cx + r, cy + r)])
            
        elif tag == 'line':
            x1, y1 = float(elem.get('x1', 0)), float(elem.get('y1', 0))
            x2, y2 = float(elem.get('x2', 0)), float(elem.get('y2', 0))
            coords.extend([(x1, y1), (x2, y2)])
            
        elif tag in ['polygon', 'polyline']:
            nums = [float(n) for n in re.findall(r'-?\d+\.?\d*', elem.get('points', ''))]
            coords.extend([(nums[i], nums[i+1]) for i in range(0, len(nums)-1, 2)])
            
        elif tag == 'path':
            # Approximation: extract all numbers to find extents
            nums = [float(n) for n in re.findall(r'-?\d+\.?\d*', elem.get('d', ''))]
            coords.extend([(nums[i], nums[i+1]) for i in range(0, len(nums)-1, 2)])
            
        elif tag in ['text', 'tspan']:
            if 'x' in elem.attrib and 'y' in elem.attrib:
                x_vals = re.findall(r'-?\d+\.?\d*', elem.get('x', '0'))
                y_vals = re.findall(r'-?\d+\.?\d*', elem.get('y', '0'))
                if x_vals and y_vals:
                    coords.append((float(x_vals[0]), float(y_vals[0])))

        # Recursively include children bounds for groups
        for child in elem:
            child_bounds = self._get_element_bounds(child)
            if child_bounds:
                coords.extend([(child_bounds[0], child_bounds[1]), (child_bounds[2], child_bounds[3])])

        if not coords:
            return None

        minx = min(c[0] for c in coords)
        miny = min(c[1] for c in coords)
        maxx = max(c[0] for c in coords)
        maxy = max(c[1] for c in coords)

        return (minx, miny, maxx, maxy)

    def parse_and_identify(self, file_path):
        """
        Parses the SVG and identifies candidate AOI elements.
        Also records:
        - stimulus_id
        - family
        - aoi_label
        - bbox placeholder
        """
        if not os.path.exists(file_path):
            return f"Error: {file_path} not found."

        stimulus_id, viz_type = self._get_type_info(file_path)
        prefixes = self.idiom_rules.get(viz_type, [])

        tree = ET.parse(file_path)
        root = tree.getroot()

        # Build parent map
        parent_map = {child: parent for parent in tree.iter() for child in parent}

        identified_elements = []

        for elem in root.iter():
            tag = elem.tag.split("}")[-1]
            elem_id = elem.attrib.get("id", "")
            elem_class = elem.attrib.get("class", "")

            parent = parent_map.get(elem)
            parent_id = parent.attrib.get("id", "ROOT") if parent is not None else "NONE"

            is_svg_shape_or_group = tag in [
                "path", "rect", "circle", "text", "line",
                "polygon", "polyline", "g"
            ]

            is_meaningful = self._is_meaningful(elem_id, elem_class, parent_id, prefixes)

            if is_svg_shape_or_group and is_meaningful:
                semantic_type = self._classify_semantic_type(elem_id, elem_class, parent_id, tag)
                text_content = self._get_text_content(elem) if tag == "text" else ""

                bbox = []
                bounds = self._get_element_bounds(elem)
                if bounds:
                    minx, miny, maxx, maxy = bounds
                    bbox = [minx, miny, maxx - minx, maxy - miny]  # [x, y, width, height]

                identified_elements.append({
                    "stimulus_id": stimulus_id,
                    "aoi_id": elem_id,
                    "aoi_label": self._get_clean_label(elem_id, text_content),
                    "aoi_type": semantic_type,
                    "family": self._get_family(semantic_type),
                    "tag": tag,
                    "parent_id": parent_id,
                    "bbox": bbox,
                    "text": text_content
                })

        return identified_elements

In [4]:
pipeline = AOIPipeline()
base_path = "data/svg/"

test_files = ["B-a-1.svg","B-a-2.svg"]

for f_name in test_files:
    file_path = os.path.join(base_path, f_name)
    results = pipeline.parse_and_identify(file_path)

    if isinstance(results, list):
        stim_id, v_type = pipeline._get_type_info(f_name)
        print(f"\n=== FILE: {f_name} | Type: {v_type} ===")
        print(f"Found {len(results)} identified elements.")

        print("\nSample results:")
        for r in results:
            print(r)

        print("-" * 80)
    else:
        print(results)


=== FILE: B-a-1.svg | Type: Ba ===
Found 81 identified elements.

Sample results:
{'stimulus_id': 'B-a-1', 'aoi_id': 'reference-axis-xAxis', 'aoi_label': 'xAxis', 'aoi_type': 'axis', 'family': 'chart_frame', 'tag': 'g', 'parent_id': 'data-container', 'bbox': [-7.8, 0.0, 393.37, 1040.0], 'text': ''}
{'stimulus_id': 'B-a-1', 'aoi_id': 'axisLine-xAxis', 'aoi_label': 'xAxis', 'aoi_type': 'axis', 'family': 'chart_frame', 'tag': 'path', 'parent_id': 'reference-axis-xAxis', 'bbox': [-7.8, 875.13, 393.37, 164.87], 'text': ''}
{'stimulus_id': 'B-a-1', 'aoi_id': 'axisTicks-xAxis', 'aoi_label': 'xAxis', 'aoi_type': 'axis', 'family': 'chart_frame', 'tag': 'g', 'parent_id': 'reference-axis-xAxis', 'bbox': [0.0, 0.0, 0.0, 0.0], 'text': ''}
{'stimulus_id': 'B-a-1', 'aoi_id': 'axis-label-container', 'aoi_label': 'container', 'aoi_type': 'label', 'family': 'annotations', 'tag': 'g', 'parent_id': 'axisTicks-xAxis', 'bbox': [0.0, 0.0, 0.0, 0.0], 'text': ''}
{'stimulus_id': 'B-a-1', 'aoi_id': '', 'aoi_la